# RoBERTa-large versus TAPT RoBERTa-large: results comparison (seed 13)

This is a **results-only notebook**. It reads the completed experiment artifacts and does not train or download a model. Its purpose is to explain what was compared, display the most important results, and make the metrics easier to interpret.

## Research question and design

The experiment asked whether task-adaptive pretraining (TAPT) on the reappraisal responses improves downstream prediction of eight reappraisal strategies. Standard RoBERTa-large was compared with the same encoder after masked-language-model adaptation.

The comparison was controlled: both classifiers used seed 13, the same 424/91/91 train-validation-test split, first-token pooling, batch size 4, learning rate `1e-5`, dropout 0.5, a linear scheduler, at most 30 epochs, and early-stopping patience 6. The only intended difference was the encoder initialization.

Targets were **soft labels**: for each strategy, the target is the proportion of valid annotators who marked it as present. A target of `0.60`, for example, means that 60% of the annotators selected that strategy. The classifier predicts eight values between 0 and 1.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_DIR = next(
    (root for root in [Path.cwd(), *Path.cwd().parents]
     if (root / 'Code' / 'BERT_models' / 'experiments' / 'tapt' / 'downstream_comparison').exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not locate the organized TAPT comparison directory.')

RESULTS_DIR = PROJECT_DIR / 'Code' / 'BERT_models' / 'experiments' / 'tapt' / 'downstream_comparison'
SUMMARY_PATH = RESULTS_DIR / 'roberta_large_tapt_comparison_seed13.csv'
BASE_SOFT_PATH = RESULTS_DIR / 'roberta_large_cls_lr_1e_05_random_seed13_soft_metrics.csv'
TAPT_SOFT_PATH = RESULTS_DIR / 'roberta_large_tapt_cls_lr_1e_05_random_seed13_soft_metrics.csv'

for required_path in [SUMMARY_PATH, BASE_SOFT_PATH, TAPT_SOFT_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing completed result: {required_path}')

summary = pd.read_csv(SUMMARY_PATH)
summary['model'] = summary['experiment_name'].map(
    lambda name: 'TAPT RoBERTa-large' if '_tapt_' in name else 'Standard RoBERTa-large'
)
baseline = summary.loc[summary['model'] == 'Standard RoBERTa-large'].iloc[-1]
tapt = summary.loc[summary['model'] == 'TAPT RoBERTa-large'].iloc[-1]

print(f'Results directory: {RESULTS_DIR}')
print(f'Loaded {len(summary)} completed comparison runs.')

## How to read the metrics

- **Best validation loss:** binomial negative log-likelihood at the selected epoch. Lower is better. It is useful for checkpoint selection but is not an accuracy percentage.
- **Macro MAE:** average absolute distance between predicted and observed annotator proportions, averaged equally across the eight strategies. `0.133` means an average error of about 13.3 percentage points. Lower is better.
- **Macro RMSE:** another prediction-error measure that penalizes large mistakes more strongly than MAE. Lower is better.
- **Mean Pearson:** average linear correlation between predictions and true soft scores across strategies. Higher is better. It measures whether scores move together, not whether probabilities are perfectly calibrated.
- **Mean Spearman:** average rank correlation. Higher is better. It measures whether the model orders low- and high-scoring examples correctly.
- **Macro-F1:** harmonic mean of precision and recall after converting probabilities into present/absent decisions. Thresholds were selected separately for each strategy on the validation set and then evaluated on the test set. Higher is better.

The binary F1 analyses use three definitions: **weak presence** means at least one positive annotator vote, **majority** means more than half, and **strong presence** means at least 80%. Scores from different definitions should not be compared directly because their class frequencies differ.

In [ ]:
metric_specs = [
    ('best_val_loss', 'Best validation loss', 'lower'),
    ('macro_mae', 'Test macro MAE', 'lower'),
    ('macro_rmse', 'Test macro RMSE', 'lower'),
    ('mean_pearson', 'Test mean Pearson', 'higher'),
    ('mean_spearman', 'Test mean Spearman', 'higher'),
    ('weak_presence_tuned_macro_f1', 'Weak-presence macro-F1', 'higher'),
    ('majority_tuned_macro_f1', 'Majority macro-F1', 'higher'),
    ('strong_presence_tuned_macro_f1', 'Strong-presence macro-F1', 'higher'),
]

comparison_rows = []
for column, label, direction in metric_specs:
    standard_value = float(baseline[column])
    tapt_value = float(tapt[column])
    difference = tapt_value - standard_value
    if column == 'majority_tuned_macro_f1' and abs(difference) < 0.002:
        result = 'Effectively tied'
    elif (direction == 'lower' and tapt_value < standard_value) or (direction == 'higher' and tapt_value > standard_value):
        result = 'TAPT'
    else:
        result = 'Standard'
    comparison_rows.append({
        'Metric': label,
        'Preferred direction': direction,
        'Standard RoBERTa': standard_value,
        'TAPT RoBERTa': tapt_value,
        'TAPT minus standard': difference,
        'Numerically better': result,
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison.style.format({
    'Standard RoBERTa': '{:.4f}',
    'TAPT RoBERTa': '{:.4f}',
    'TAPT minus standard': '{:+.4f}',
}))

print(f"Standard: best epoch {int(baseline['best_epoch'])}; early stopping occurred after epoch 12.")
print(f"TAPT: best epoch {int(tapt['best_epoch'])}; early stopping occurred after epoch 14.")

In [ ]:
colors = ['#4C78A8', '#F58518']
models = ['Standard RoBERTa-large', 'TAPT RoBERTa-large']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

error_metrics = ['macro_mae', 'macro_rmse']
error_labels = ['Macro MAE', 'Macro RMSE']
error_values = [[baseline[m] for m in error_metrics], [tapt[m] for m in error_metrics]]
pd.DataFrame(error_values, index=models, columns=error_labels).T.plot.bar(ax=axes[0], color=colors)
axes[0].set_title('Soft-label error (lower is better)')
axes[0].set_ylim(0, 0.22)

corr_metrics = ['mean_pearson', 'mean_spearman']
corr_labels = ['Mean Pearson', 'Mean Spearman']
corr_values = [[baseline[m] for m in corr_metrics], [tapt[m] for m in corr_metrics]]
pd.DataFrame(corr_values, index=models, columns=corr_labels).T.plot.bar(ax=axes[1], color=colors)
axes[1].set_title('Soft-label association (higher is better)')
axes[1].set_ylim(0, 0.7)

f1_metrics = [
    'weak_presence_tuned_macro_f1',
    'majority_tuned_macro_f1',
    'strong_presence_tuned_macro_f1',
]
f1_labels = ['Weak', 'Majority', 'Strong']
f1_values = [[baseline[m] for m in f1_metrics], [tapt[m] for m in f1_metrics]]
pd.DataFrame(f1_values, index=models, columns=f1_labels).T.plot.bar(ax=axes[2], color=colors)
axes[2].set_title('Tuned macro-F1 (higher is better)')
axes[2].set_ylim(0, 0.8)

for axis in axes:
    axis.set_xlabel('')
    axis.tick_params(axis='x', rotation=0)
    axis.grid(axis='y', alpha=0.25)
    if axis.get_legend() is not None:
        axis.get_legend().remove()

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False)
fig.suptitle('Seed-13 downstream comparison', y=1.05, fontsize=14)
fig.tight_layout()
plt.show()

## Strategy-level results

The averages can hide different effects for individual strategies. The next table shows the test metrics separately for all eight strategies.

In [ ]:
standard_soft = pd.read_csv(BASE_SOFT_PATH)
tapt_soft = pd.read_csv(TAPT_SOFT_PATH)
strategy_comparison = standard_soft.merge(
    tapt_soft,
    on=['label', 'strategy'],
    suffixes=('_standard', '_tapt'),
)
strategy_comparison['mae_change_tapt_minus_standard'] = (
    strategy_comparison['mae_tapt'] - strategy_comparison['mae_standard']
)
strategy_comparison['pearson_change_tapt_minus_standard'] = (
    strategy_comparison['pearson_tapt'] - strategy_comparison['pearson_standard']
)
display(strategy_comparison[[
    'strategy',
    'mae_standard', 'mae_tapt', 'mae_change_tapt_minus_standard',
    'pearson_standard', 'pearson_tapt', 'pearson_change_tapt_minus_standard',
]].style.format(precision=4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].barh(
    strategy_comparison['strategy'],
    strategy_comparison['mae_change_tapt_minus_standard'],
    color=['#54A24B' if value < 0 else '#E45756' for value in strategy_comparison['mae_change_tapt_minus_standard']],
)
axes[0].axvline(0, color='black', linewidth=1)
axes[0].set_title('Change in MAE')
axes[0].set_xlabel('TAPT minus standard (negative favors TAPT)')

axes[1].barh(
    strategy_comparison['strategy'],
    strategy_comparison['pearson_change_tapt_minus_standard'],
    color=['#54A24B' if value > 0 else '#E45756' for value in strategy_comparison['pearson_change_tapt_minus_standard']],
)
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('Change in Pearson correlation')
axes[1].set_xlabel('TAPT minus standard (positive favors TAPT)')

for axis in axes:
    axis.grid(axis='x', alpha=0.25)
fig.tight_layout()
plt.show()

## Main outcome and appropriate interpretation

The TAPT run was technically successful, but it did **not** produce a clear overall downstream improvement. TAPT had slightly lower validation loss and RMSE and higher Spearman correlation. Standard RoBERTa had slightly lower MAE, higher Pearson correlation, and better weak- and strong-presence macro-F1. Majority macro-F1 was effectively tied (`0.4530` versus `0.4545`).

The most defensible conclusion is: **for this seed-13 split, TAPT and standard RoBERTa performed similarly, with small and inconsistent differences across metrics. There is no clear evidence that TAPT improved classification.** This does not establish that TAPT never helps. Only one seed and one test split were evaluated, so there is no estimate of variation across runs.

For stronger evidence, repeat the matched comparison across multiple seeds and report the mean and standard deviation for each metric. A leave-one-situation-out comparison would answer a different and stricter question: whether either model generalizes to a completely unseen situation.

## Source artifacts

All values in this notebook are loaded from `Code/BERT_models/experiments/tapt/downstream_comparison`. The original training notebook is `notebooks/training/RoBERTa_TAPT_Comparison_Seed13.ipynb`, and the task-adaptive pretraining notebook is `notebooks/training/RoBERTa_Masked_Language_Modelling_TAPT.ipynb`.